# 03b - Tambah positif TARGETED untuk slice yang bias negatif

Masalah: kata/tanda tertentu (mis. `mbg`, `?`) di data latih hampir selalu muncul di komentar negatif, sehingga model belajar
"ada kata itu -> negatif" dan salah membaca komentar positif yang memuat kata tsb ("mbg program bagus").

Pendekatan (beda dengan sintetis umum sebelumnya):
1. **Tampilkan** kata/tanda yang bias negatif di data latih (berdasarkan angka), lalu **kamu pilih** yang bersifat topik/netral (mis. `mbg`, `?`), bukan kata bermakna negatif.
2. Hitung berapa positif yang kurang agar tiap kata mendekati base rate (`TARGET_SHARE`), dengan batas atas.
3. **LLM menulis positif baru yang WAJIB memuat kata itu**, memakai contoh nyata dari data latih sebagai few-shot, dengan 4 gaya.
4. **Filter:** harus memuat kata, bukan duplikat/mirip data latih-val-test, lalu lolos pemeriksaan sikap oleh LLM kedua (`positif`).
5. Simpan ke `targeted_positives.csv` untuk **ditinjau manual** (kolom `dipakai`), lalu diuji di notebook 04 (`SYN_MODE="targeted"`).

Aturan anti-bocor: contoh few-shot HANYA dari data latih (val dikeluarkan dengan split yang sama persis dengan notebook 04); test tidak dipakai sebagai contoh,
dan hasil generate yang mirip komentar test dibuang.

In [1]:
import re, json, math, random, time, subprocess
from pathlib import Path
from collections import Counter
import numpy as np, pandas as pd, requests

LLM_MODEL  = "qwen2.5:7b"
OLLAMA_URL = "http://localhost:11434"
SEED, VAL_SEED = 42, 42            # VAL_SEED HARUS sama dengan notebook 04
DATASET_NAME, DATA_DIR_OVERRIDE = "labels_shafwa_hans_agreement", None

# ---- pencarian kata bias ----
MIN_COUNT  = 15      # kata harus muncul di >= sekian komentar latih asli
SKEW_RATIO = 0.5     # bias-negatif kalau %positif <= 50% dari base rate
AUTO_TOP   = 0       # 0 = kata fokus dipilih MANUAL (disarankan). >0 = tambah sekian kata teratas otomatis
FORCE_KEYWORDS, EXTRA_KEYWORDS, DROP_KEYWORDS = ["mbg", "?"], [], []

# ---- jumlah generate ----
TARGET_SHARE    = 0.25   # target %positif per kata setelah ditambah
MAX_PER_KEYWORD = 60
MAX_TOTAL       = 150    # batas total (jaga supaya domain shift kecil)
BATCH_K, MAX_TRIES = 8, 40

# ---- pengaman suhu GPU (Ollama memakai GPU) ----
PAUSE_S, TEMP_PAUSE, TEMP_RESUME = 2, 80, 72

STYLES = ["pujian langsung yang singkat",
          "dukungan dalam bentuk kalimat tanya (tetap jelas mendukung)",
          "membela dari kritik secara sopan, tanpa menyerang siapa pun",
          "santai atau bercanda tetapi tetap mendukung"]
random.seed(SEED); np.random.seed(SEED)

In [2]:
def find_data_dir():
    if DATA_DIR_OVERRIDE:
        p = Path(DATA_DIR_OVERRIDE)
        if (p/"train_balanced.csv").exists(): return p
        raise FileNotFoundError(f"train_balanced.csv tidak ada di {p}")
    here = Path.cwd().resolve(); found = []
    for base in [here] + list(here.parents):
        root = base / "data" / "praprocessing"
        if root.is_dir():
            found += [d for d in root.iterdir() if (d/"train_balanced.csv").exists() and (d/"test_final.csv").exists()]
        if found: break
    if not found:
        found = [p.parent for p in here.rglob("train_balanced.csv") if (p.parent/"test_final.csv").exists()]
    if not found: raise FileNotFoundError(f"train_balanced.csv tidak ketemu. Folder kerja: {here}")
    pref = [d for d in found if d.name == DATASET_NAME]
    return pref[0] if pref else max(found, key=lambda d: (d/"train_balanced.csv").stat().st_mtime)

DATA_DIR = find_data_dir()
print("DATA_DIR:", DATA_DIR)

DATA_DIR: C:\Users\nesha\VSCODE_ALL\PBAAnalysisSentiment\data\praprocessing\labels_shafwa_hans_agreement


## 1. Muat data & ulangi split val persis seperti notebook 04

In [3]:
def norm(t): return re.sub(r"\s+", " ", str(t).lower()).strip()
def toks(t): return set(re.findall(r"\w+|\?", norm(t)))

tr = pd.read_csv(DATA_DIR/"train_balanced.csv", encoding="utf-8")
te = pd.read_csv(DATA_DIR/"test_final.csv", encoding="utf-8")
if "sumber" not in tr.columns: tr["sumber"] = "asli"
tr["label"] = tr["final_label"].map({"negatif": 0, "positif": 1})

real = tr[tr["sumber"] == "asli"]
val = pd.concat([real[real["label"] == k].sample(frac=0.12, random_state=VAL_SEED) for k in (0, 1)])
train_real = real.drop(val.index).reset_index(drop=True)      # asli, tanpa val -> satu-satunya sumber contoh
train_real["toks"] = train_real["comment_text"].map(toks)
base = train_real["label"].mean()
print(f"Data latih asli (tanpa val): {len(train_real)} | positif {int(train_real.label.sum())} ({base:.0%}) | val dikeluarkan: {len(val)} | test: {len(te)}")

Data latih asli (tanpa val): 957 | positif 297 (31%) | val dikeluarkan: 131 | test: 446


## 2. Kata/tanda yang bias negatif

In [4]:
cnt, pos = Counter(), Counter()
for ts, y in zip(train_real["toks"], train_real["label"]):
    for t in ts: cnt[t] += 1; pos[t] += y
M = 5   # smoothing ke base rate
rows = [(t, n, pos[t], (pos[t] + base*M)/(n + M), n*base - pos[t]) for t, n in cnt.items() if n >= MIN_COUNT]
bias = pd.DataFrame(rows, columns=["token", "n", "n_pos", "share_pos", "positif_hilang"]).sort_values("positif_hilang", ascending=False)
skew = bias[bias["share_pos"] <= base * SKEW_RATIO]

print(f"base rate positif = {base:.0%}. Kata bias-negatif teratas (positif_hilang = positif yang 'seharusnya ada' tapi tidak):")
print(skew.head(20).round(3).to_string(index=False))

FOCUS = [k for k in dict.fromkeys(FORCE_KEYWORDS + EXTRA_KEYWORDS + skew.head(AUTO_TOP)["token"].tolist()) if k not in DROP_KEYWORDS]
print("\nKata fokus:", FOCUS)
print("PENTING: daftar di atas hanya SARAN. Pilih (lewat FORCE/EXTRA_KEYWORDS) kata yang bersifat TOPIK atau tanda baca netral, mis. 'mbg', '?'.")
print("JANGAN pilih kata yang memang bermakna negatif (mis. 'gagal', 'kecewa', 'bodoh'): di situ bias ke negatif itu wajar, dan memaksa positif memuat kata itu malah menyesatkan model.")

base rate positif = 31%. Kata bias-negatif teratas (positif_hilang = positif yang 'seharusnya ada' tapi tidak):
  token  n  n_pos  share_pos  positif_hilang
laporan 64      0      0.022          19.862
    mbg 80      8      0.112          16.828
 nuklir 52      1      0.045          15.138
      ? 62      6      0.113          13.241
  dapat 34      1      0.065           9.552
   iran 21      0      0.060           6.517
  bahas 18      0      0.067           5.586

Kata fokus: ['mbg', '?']
PENTING: daftar di atas hanya SARAN. Pilih (lewat FORCE/EXTRA_KEYWORDS) kata yang bersifat TOPIK atau tanda baca netral, mis. 'mbg', '?'.
JANGAN pilih kata yang memang bermakna negatif (mis. 'gagal', 'kecewa', 'bodoh'): di situ bias ke negatif itu wajar, dan memaksa positif memuat kata itu malah menyesatkan model.


## 3. Rencana jumlah generate per kata

In [5]:
plan = []
for kw in FOCUS:
    sub = train_real[train_real["toks"].map(lambda s: kw in s)]
    n, npos = len(sub), int(sub["label"].sum())
    g = max(0, math.ceil((TARGET_SHARE*n - npos) / (1 - TARGET_SHARE)))
    plan.append({"kw": kw, "n_komentar": n, "n_positif": npos, "share_sekarang": npos/max(n, 1), "g": min(g, MAX_PER_KEYWORD)})
plan = pd.DataFrame(plan)
if plan["g"].sum() > MAX_TOTAL:
    plan["g"] = (plan["g"] * MAX_TOTAL / plan["g"].sum()).round().astype(int)
plan["share_target"] = (plan["n_positif"] + plan["g"]) / (plan["n_komentar"] + plan["g"]).clip(lower=1)
print(plan.round(2).to_string(index=False))
print("\nTotal akan digenerate:", int(plan["g"].sum()), "| (maks", MAX_TOTAL, ")")

 kw  n_komentar  n_positif  share_sekarang  g  share_target
mbg          80          8             0.1 16          0.25
  ?          62          6             0.1 13          0.25

Total akan digenerate: 29 | (maks 150 )


## 4. Helper LLM + filter

In [6]:
GEN_PROMPT = """Kamu membantu membuat data latih untuk klasifikasi sikap komentar TikTok berbahasa Indonesia tentang Presiden Prabowo.
Berikut contoh komentar ASLI yang sikapnya MENDUKUNG (positif):
{ex}

Tulis {k} komentar BARU dengan sikap MENDUKUNG Prabowo atau programnya. Aturan:
- setiap komentar WAJIB mengandung kata/tanda '{kw}'
- gaya: {gaya}
- bahasa santai seperti komentar TikTok asli, boleh singkat, boleh slang/emoji
- jangan menyerang atau menghina orang atau kelompok tertentu; jangan mengarang fakta atau angka
- beragam topik dan kalimat, jangan meniru contoh kata per kata
- satu komentar per baris, tanpa nomor dan tanpa penjelasan"""

CLS_PROMPT = """Komentar TikTok: "{t}"
Apa sikap komentar ini terhadap Presiden Prabowo atau programnya? Jawab SATU kata saja: positif, negatif, atau netral."""

def gpu_temp():
    try:
        return int(subprocess.run(["nvidia-smi", "--query-gpu=temperature.gpu", "--format=csv,noheader,nounits"],
                                  capture_output=True, text=True, timeout=5).stdout.strip().splitlines()[0])
    except Exception: return None

def cool_guard():
    t = gpu_temp()
    if t is not None and t >= TEMP_PAUSE:
        print(f"\n  GPU {t}C, jeda sampai <= {TEMP_RESUME}C ...", end="")
        while True:
            time.sleep(10); t = gpu_temp()
            if t is None or t <= TEMP_RESUME: break
        print(f" lanjut ({t}C)")

def llm(prompt, temperature=0.9, timeout=300):
    r = requests.post(f"{OLLAMA_URL}/api/generate", timeout=timeout,
                      json={"model": LLM_MODEL, "prompt": prompt, "stream": False,
                            "options": {"temperature": temperature, "top_p": 0.95}})
    r.raise_for_status(); return r.json()["response"]

def clean_line(s):
    return re.sub(r"^\s*(?:[-*•]|\d+[.)])\s*", "", s).strip().strip('"').strip()

def stance(c):
    a = llm(CLS_PROMPT.format(t=c), temperature=0.0, timeout=120).strip().lower()
    return "positif" if a.startswith("positif") else ("negatif" if a.startswith("negatif") else "netral")

def jacc(a, b): return len(a & b) / max(len(a | b), 1)

seen = set(map(norm, tr["comment_text"])) | set(map(norm, te["comment_text"]))
ref_sets = [toks(t) for t in list(tr["comment_text"]) + list(te["comment_text"])]   # dedupe terhadap SEMUA (latih+val+test)
def near_dup(ts, thr=0.8): return any(jacc(ts, r) >= thr for r in ref_sets)
print("Ollama:", "OK" if requests.get(f"{OLLAMA_URL}/api/tags", timeout=5).ok else "TIDAK terjangkau")

Ollama: OK


## 5. Generate (aman diulang; hasil akhir ditulis di langkah 6)

In [7]:
generated, stats = [], []
for _, row in plan.iterrows():
    kw, need = row["kw"], int(row["g"])
    if need <= 0: continue
    pool = train_real[(train_real["label"] == 1) & train_real["toks"].map(lambda s: kw in s)]["comment_text"].tolist()
    if len(pool) < 3:    # contoh positif dengan kata ini terlalu sedikit -> tambah positif asli lain
        pool += train_real[train_real["label"] == 1]["comment_text"].sample(5, random_state=SEED).tolist()
    got, tries, rej = [], 0, Counter()
    while len(got) < need and tries < MAX_TRIES:
        gaya = STYLES[tries % len(STYLES)]; tries += 1
        ex = "\n".join(f"- {e}" for e in random.sample(pool, min(5, len(pool))))
        try:
            lines = [clean_line(l) for l in llm(GEN_PROMPT.format(ex=ex, k=BATCH_K, kw=kw, gaya=gaya)).splitlines()]
        except Exception as e:
            print("  gagal batch:", e); continue
        for c in lines:
            ts = toks(c)
            if kw not in ts:                  rej["tanpa kata"] += 1; continue
            if not (5 <= len(c) <= 250):      rej["panjang"] += 1; continue
            if norm(c) in seen or near_dup(ts): rej["duplikat/mirip"] += 1; continue
            try: st = stance(c)
            except Exception: rej["error cek"] += 1; continue
            if st != "positif":               rej[f"sikap={st}"] += 1; continue
            got.append({"comment_text": c, "final_label": "positif", "sumber": "targeted", "keyword": kw, "gaya": gaya, "dipakai": 1})
            seen.add(norm(c)); ref_sets.append(ts)
            if len(got) >= need: break
        cool_guard(); time.sleep(PAUSE_S)
        print(f"\r  '{kw}': {len(got)}/{need} (batch {tries})   ", end="")
    print(f"\n  '{kw}' selesai: {len(got)}/{need} | ditolak: {dict(rej)}")
    generated += got
print("\nTotal lolos filter:", len(generated))

  'mbg': 16/16 (batch 3)   
  'mbg' selesai: 16/16 | ditolak: {'tanpa kata': 18, 'sikap=negatif': 1, 'sikap=netral': 4}
  '?': 13/13 (batch 4)   
  '?' selesai: 13/13 | ditolak: {'tanpa kata': 30, 'sikap=netral': 10, 'sikap=negatif': 5}

Total lolos filter: 29


## 6. Tinjau & simpan

In [8]:
gen_df = pd.DataFrame(generated)
out_path = DATA_DIR / "targeted_positives.csv"
if gen_df.empty:
    print("Belum ada hasil, tidak disimpan.")
else:
    gen_df.to_csv(out_path, index=False, encoding="utf-8-sig")     # utf-8-sig: emoji aman dibuka di Excel
    print("Tersimpan:", out_path, "| baris:", len(gen_df))
    print(gen_df.groupby(["keyword", "gaya"]).size().unstack(fill_value=0))
    pd.set_option("display.max_colwidth", 120); pd.set_option("display.width", 220)
    print("\n30 contoh acak untuk dicek mata:")
    print(gen_df.sample(min(30, len(gen_df)), random_state=SEED)[["keyword", "comment_text"]].to_string(index=False))

Tersimpan: C:\Users\nesha\VSCODE_ALL\PBAAnalysisSentiment\data\praprocessing\labels_shafwa_hans_agreement\targeted_positives.csv | baris: 29
gaya     dukungan dalam bentuk kalimat tanya (tetap jelas mendukung)  \
keyword                                                                
?                                                        2             
mbg                                                      4             

gaya     membela dari kritik secara sopan, tanpa menyerang siapa pun  \
keyword                                                                
?                                                        1             
mbg                                                      4             

gaya     pujian langsung yang singkat  \
keyword                                 
?                                   7   
mbg                                 8   

gaya     santai atau bercanda tetapi tetap mendukung  
keyword                                               
?     

## Langkah berikutnya
1. **Buka `targeted_positives.csv` dan tinjau.** Set kolom `dipakai` jadi `0` untuk komentar yang bukan dukungan jelas, terdengar tidak natural, atau menyerang orang. Meninjau 100-150 baris cuma beberapa menit dan paling menentukan kualitasnya.
2. Notebook 04 (v3): set `SYN_MODE = "targeted"` atau jalankan sweep. Hanya baris `dipakai == 1` yang dimuat.
3. Bandingkan dengan `real_cw` memakai tabel perbandingan, **uji pasangan minimal**, dan laporan slice `mbg` / `?`.